# FIS ontology date keys

Ontology entity keys require String or BigInt. Preserve the original DATE relationships and add an integer yyyyMMdd key only to the date edges.

In [ ]:
for fact in ("ACTUALS", "FORECAST", "PLAN"):
    table = f"edge_{fact}_dim_date".lower()
    if "TargetDateInteger" not in spark.table(table).columns:
        spark.sql(f"ALTER TABLE {table} ADD COLUMNS (TargetDateInteger BIGINT)")
    spark.sql(f"UPDATE {table} SET TargetDateInteger = CAST(date_format(TargetId, 'yyyyMMdd') AS BIGINT)")
    invalid = spark.sql(f"SELECT COUNT(*) AS n FROM {table} WHERE TargetDateInteger IS NULL OR TargetDateInteger <> CAST(date_format(TargetId, 'yyyyMMdd') AS BIGINT)").first()["n"]
    if invalid:
        raise ValueError(f"Invalid integer ontology date keys in {table}: {invalid}")
    print(table, "validated", spark.table(table).count())
